# Tutorial 3 — Scour Hole Evolution in a Gravel-Bed Channel

## Motivation

Tutorials 1 and 2 study a channel fed with excess sediment at the upstream boundary, producing steady aggradation.  This tutorial takes a different perspective: **what happens when the bed itself contains a local perturbation — a 5 m deep scour hole — and the flow is left to reshape it?**

This scenario isolates the morphodynamic response to a bed disturbance and provides a clear test-bed for the gravitational bed diffusion feature added in V2.

## Physical Setup

| Parameter | Value |
|-----------|-------|
| Channel length | 1 500 m |
| Median grain size D₅₀ | 50 mm |
| Manning's roughness *n* | 0.039 |
| Hole depth | 5 m (Gaussian, σ = 75 m, FWHM ≈ 175 m) |
| Hole centre | x = 750 m from the outlet |
| Upstream sediment supply | None imposed — transport is computed locally |
| Boundary conditions | Fixed bed elevation at outlet and upstream end |

## Expected Physics

Over the hole, the water column deepens and velocity drops, reducing local shear stress and transport capacity.  Sediment arriving from upstream cannot be carried through the depression and deposits on the **upstream lip** of the hole.  The downstream lip, **starved of sediment**, may erode slightly.  The net result is:

1. **Rapid initial infilling** from the upstream side (dominant in the first few hours).
2. **Gradual downstream migration** of the residual depression.
3. **Asymptotic approach** to the original bed profile as the perturbation decays.

Gravitational diffusion (Talmon et al. 1995) accelerates the smoothing of sharp slope breaks at the hole edges, and its effect should be most visible during the rapid early phase.

## Simulations

| Run | Time stepping | Diffusion | Section |
|-----|--------------|-----------|---------|
| Baseline | Euler | Off | 5–6 |
| Diffusion | Euler | On (μ = 0.5) | 7–8 |

## Key V2 Features Demonstrated

- `check_advective_cfl=False` — CFL is managed by the coupling loop, not by RBD
- `use_bed_diffusion=True`, `bed_diffusion_mu=0.5` — gravitational diffusion
- `calc_max_stable_dt_*()` — post-run CFL stability analysis
- `_bed_surf__gsd_residual_max/mean` — GSD numerical health diagnostics
- `map_mean_of_link_nodes_to_link` — link-depth synchronisation fix (Landlab ≥ 2.10)

## 1. Imports

In [ ]:
import copy

import matplotlib.cm as cm
import numpy as np
from matplotlib import pyplot as plt

from landlab import imshow_grid
from landlab.components import OverlandFlow, RiverBedDynamics
from landlab.grid.mappers import map_mean_of_link_nodes_to_link
from landlab.io.esri_ascii import load as load_esri_ascii

## 2. Simulation Parameters

We reuse the same grid and flow forcing as Tutorials 1–2 (1 500 m channel, Manning's *n* = 0.039, rainfall proxy at the inlet nodes).  The only change is the hole geometry and the boundary conditions.

Snapshots are saved every **1 hour** to resolve the rapid early evolution of the hole.

In [ ]:
# Input files (same as Tutorials 1–2)
zDEM = "bedElevationDEM.asc"
gsd = np.loadtxt("bed_gsd.txt")

# Time parameters
max_dt = 1                                 # Time step [s]
sim_max_t = 3 * 86400 + max_dt             # Morphodynamic simulation [s] (3 days)
sim_max_t_spinup = 86400 + max_dt          # Hydraulic spin-up [s] (1 day)
save_interval = 3600                       # Save profile every 1 hour [s]

# Manning's roughness (same as Tutorials 1–2)
n_manning = 0.03874

# Rainfall proxy nodes (drives the flow, same as Tutorials 1–2)
in_n = np.array((129, 130))

# Hole parameters
hole_center_x = 750.0   # Streamwise distance from outlet [m]
hole_depth = 5.0         # Maximum excavation depth [m]
hole_sigma = 75.0        # Gaussian standard deviation [m] (FWHM ≈ 175 m)

# Grid topology (same as Tutorials 1–2)
fixed_outlet_nodes = np.array((1, 2, 5, 6))      # Outlet (bottom rows)
ghost_cell_ids = np.arange(128, 136)              # Top 2 ghost rows
sample_node_id = np.arange(5, 126, 4)             # Column-1 profile extraction

# Fixed upstream nodes: rows 28–31 (x ≈ 1 350–1 500 m)
# Well upstream of the hole — prevents the inlet from eroding
fixed_upstream_nodes = np.arange(112, 128)

print("=" * 60)
print("Tutorial 3 — Scour Hole Evolution")
print("=" * 60)
print(f"  Hole : {hole_depth:.0f} m deep, centred at x = {hole_center_x:.0f} m, "
      f"σ = {hole_sigma:.0f} m")
print(f"  Run  : {sim_max_t / 86400:.0f} days, dt = {max_dt} s")
print()

## 3. Grid Setup — Load DEM and Excavate the Hole

We load the same DEM used in all tutorials and dig a Gaussian depression into the bed.  The hole is defined in streamwise coordinates (x = 0 at the outlet, x = 1 500 at the inlet) and mapped to the grid's y-coordinate system.

Closed-boundary nodes (the side walls at z = 45 m) are restored after the excavation to avoid breaking the boundary condition.

In [ ]:
with open(zDEM) as f:
    rmg = load_esri_ascii(f, name="topographic__elevation")
z = rmg.at_node["topographic__elevation"]

n_col = rmg.number_of_node_columns
n_row = rmg.number_of_node_rows

# Add required fields before boundary conditions
rmg.add_zeros("surface_water__depth", at="node")

# Set boundary conditions (outlet at nodes 1,2; edges closed at z = 45)
rmg.set_watershed_boundary_condition_outlet_id([1, 2], z, 45.0)

# Save the original (hole-free) profile for reference
z_original = z.copy()

# ── Dig the Gaussian hole ────────────────────────────────────────────
# Grid y-coordinate: y = 0 at bottom row, y increases upstream.
# Streamwise x in the tutorial goes from 0 (outlet) to 1500 (inlet).
# Profile sample starts at row 1 (y = dx) → x = 0.
# Therefore: y_grid = x_streamwise + dx
hole_center_y = hole_center_x + rmg.dx   # grid y-coordinate of hole centre

hole_profile = hole_depth * np.exp(
    -((rmg.y_of_node - hole_center_y) ** 2) / (2.0 * hole_sigma ** 2)
)
z -= hole_profile

# Restore closed-boundary nodes (edges at z = 45) — the hole must not
# affect them, otherwise the boundary condition breaks.
closed_ids = np.where(rmg.status_at_node == 4)[0]
z[closed_ids] = z_original[closed_ids]

# Profile coordinates
x_profile = np.linspace(0, 1500, sample_node_id.shape[0])
z_profile_with_hole = z[sample_node_id].copy()
z_profile_original = z_original[sample_node_id].copy()

print(f"Grid : {n_row} rows × {n_col} cols, dx = {rmg.dx:.0f} m")
print(f"Hole applied.  Max depression: {hole_profile.max():.2f} m")

### Visualisation — initial bed profile

The plot below shows the original (undisturbed) bed profile and the excavated profile.  The shaded region represents the volume of material removed.  Note the smooth Gaussian shape of the depression, which avoids sharp corners that could cause numerical artifacts.

In [ ]:
# ── Figure 1 — Initial condition ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x_profile, z_profile_original, "k-", lw=1.5, label="Original bed (no hole)")
ax.plot(x_profile, z_profile_with_hole, "r-", lw=2, label="Bed with hole")
ax.fill_between(
    x_profile, z_profile_with_hole, z_profile_original,
    alpha=0.25, color="red", label="Excavated volume",
)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Bed elevation [m]")
ax.set_title("Initial condition: scour hole in gravel-bed channel")
ax.legend()
ax.set_xlim(0, 1500)
plt.tight_layout()
plt.savefig("fig_t3_01_initial_condition.png", dpi=150)
plt.show()
print("Saved: fig_t3_01_initial_condition.png\n")

## 4. Hydraulic Spin-Up on the Original Bed

A key challenge when introducing a bed perturbation is avoiding a **hydraulic transient** at *t* = 0.  If we simply set uniform water depth over the hole, the sudden 5 m drop in bed elevation creates a "waterfall" that generates shock waves in the shallow-water solver.

**Our strategy:** we run a 1-day hydraulic spin-up on the *original* (hole-free) bed using a separate scratch grid.  This produces a smooth, quasi-steady water surface elevation (WSE).  We then transplant this WSE onto the main grid — which has the hole — and compute the water depth as:

$$h = \text{WSE}_{\text{smooth}} - z_{\text{with hole}}$$

This gives a continuous water surface that simply deepens over the depression, with no hydraulic jump at *t* = 0.

In [ ]:
print("Hydraulic spin-up on original bed (1 day)...")

with open(zDEM) as f:
    rmg_su = load_esri_ascii(f, name="topographic__elevation")
z_su = rmg_su.at_node["topographic__elevation"]
rmg_su.add_zeros("surface_water__depth", at="node")
rmg_su.set_watershed_boundary_condition_outlet_id([1, 2], z_su, 45.0)

of_su = OverlandFlow(
    rmg_su, h_init=0.001, mannings_n=n_manning, rainfall_intensity=0.0,
)
of_su._rainfall_intensity = np.zeros_like(z_su, dtype=float)
of_su._rainfall_intensity[in_n] = 0.02

rmg_su.add_zeros("surface_water__velocity", at="node")
rmg_su.add_zeros("surface_water__velocity", at="link")
rmg_su["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg_su, "surface_water__depth"
)

t0 = 0
progress0 = 0
while t0 < sim_max_t_spinup:
    of_su.overland_flow(dt=max_dt)
    rmg_su["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_su, "surface_water__depth"
    )
    t0 += max_dt
    progress = int((t0 / sim_max_t_spinup) * 100)
    if progress > progress0 + 1:
        print(f"\r  Spin-up: [{progress}%]", end="")
        progress0 = progress

# Record the equilibrium water surface elevation (WSE = z + h)
wse_equilibrium = z_su + rmg_su.at_node["surface_water__depth"]
print("\n  Spin-up complete.\n")

### Transplant the smooth water surface onto the hole

With the equilibrium WSE recorded, we compute the initial water depth over the excavated bed.  The `np.maximum(..., 0.001)` guard ensures a minimum thin film of water everywhere, preventing division-by-zero in the shallow-water solver.

In [ ]:
h_over_hole = np.maximum(wse_equilibrium - z, 0.001)
rmg.at_node["surface_water__depth"][:] = h_over_hole

### Visualisation — water surface initialisation check

The left panel confirms that the water surface (blue dashed) is smooth and continuous over the hole — it does not follow the bed depression.  The right panel shows the corresponding water depth profile: a clear peak of ~5 m over the hole centre, tapering smoothly to the ambient depth (~0.3–0.5 m) outside the depression.

In [ ]:
# ── Figure 2 — Water surface check ───────────────────────────────────
wse_check = z[sample_node_id] + h_over_hole[sample_node_id]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_profile, z_profile_original, "k:", lw=1, alpha=0.5, label="Original bed")
ax.plot(x_profile, z[sample_node_id], "r-", lw=2, label="Bed (with hole)")
ax.plot(x_profile, wse_check, "b--", lw=1.5, label="Water surface")
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Elevation [m]")
ax.set_title("Initial water surface over the scour hole")
ax.legend(fontsize=9)
ax.set_xlim(0, 1500)

ax = axes[1]
ax.plot(x_profile, h_over_hole[sample_node_id], "b-", lw=2)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Water depth [m]")
ax.set_title("Initial water depth profile")
ax.set_xlim(0, 1500)

plt.tight_layout()
plt.savefig("fig_t3_02_initial_water_surface.png", dpi=150)
plt.show()
print("Saved: fig_t3_02_initial_water_surface.png\n")

## 5. Baseline Simulation — Setup

We instantiate OverlandFlow and RiverBedDynamics for the baseline run (forward Euler, no diffusion).

**Important details:**
- `OverlandFlow.__init__` resets `surface_water__depth` to `h_init` everywhere, so we must **restore** our carefully initialised depth field afterward.
- **No sediment supply** is imposed — `sed_transp__bedload_rate_fix_link` is not set.  Transport is computed entirely from the local hydraulics.
- Both the **outlet** (nodes 1, 2, 5, 6) and the **upstream boundary** (nodes 112–127) have fixed bed elevation.  This keeps the ends pinned and allows only the hole region to evolve freely.
- `check_advective_cfl=False` because in a coupled run the timestep is controlled by OverlandFlow, not by the morphodynamic CFL condition.

In [ ]:
# Store the pre-hole topography for later plotting
rmg["node"]["topographic__elevation_original"] = z_original.copy()

# ── OverlandFlow ──────────────────────────────────────────────────────
of = OverlandFlow(
    rmg, h_init=0.001, mannings_n=n_manning, rainfall_intensity=0.0,
)
of._rainfall_intensity = np.zeros_like(z, dtype=float)
of._rainfall_intensity[in_n] = 0.02

# Restore our carefully initialised water depth (OverlandFlow.__init__
# resets it to h_init = 0.001 everywhere)
rmg.at_node["surface_water__depth"][:] = h_over_hole

rmg.add_zeros("surface_water__velocity", at="node")
rmg.add_zeros("surface_water__velocity", at="link")
rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg, "surface_water__depth"
)

# ── Fixed-elevation mask ──────────────────────────────────────────────
# Outlet nodes + upstream rows → bed elevation does not change there.
# No imposed sediment supply — transport is purely self-generated.
fixed_nodes = np.zeros(rmg.number_of_nodes)
fixed_nodes[fixed_outlet_nodes] = 1
fixed_nodes[fixed_upstream_nodes] = 1

# ── RiverBedDynamics ──────────────────────────────────────────────────
rbd = RiverBedDynamics(
    rmg,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_nodes,
    check_advective_cfl=False,        # CFL managed by coupling loop
)

# Ghost-cell setup (zero-gradient upstream BC)
n_row_ghost = int(ghost_cell_ids.shape[0] / n_col)
calc_node_id = np.reshape(ghost_cell_ids, (n_row_ghost, n_col))

print("Components instantiated (baseline, no diffusion).")
print(f"  Fixed-elevation nodes: outlet {fixed_outlet_nodes.tolist()}")
print(f"                        upstream {fixed_upstream_nodes[0]}–"
      f"{fixed_upstream_nodes[-1]}")

### Brief hydraulic adjustment (30 minutes)

The spin-up was performed on the original (hole-free) bed.  Now that the hole is present, we run a short 30-minute hydraulic adjustment to let the velocity field and discharge adapt to the new geometry before morphodynamics begin.  This is much shorter than the full spin-up because the water surface is already close to equilibrium — only the local flow pattern over the hole needs to adjust.

In [ ]:
print("\nHydraulic adjustment with hole (30 min)...")
t_adj = 0
t_adj_max = 1800   # 30 minutes

while t_adj < t_adj_max:
    of.overland_flow(dt=max_dt)
    rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg, "surface_water__depth"
    )
    t_adj += max_dt

rmg["link"]["surface_water__velocity"] = (
    rmg["link"]["surface_water__discharge"] / rmg["link"]["surface_water__depth"]
)
print("  Adjustment complete.\n")

### Run the baseline simulation (3 days)

The coupled loop follows the same structure as Tutorial 2:

1. Record previous-step velocity (for the unsteady shear stress term)
2. Advance OverlandFlow by one timestep
3. **Remap node depths → links** (`map_mean_of_link_nodes_to_link`) — critical fix for Landlab ≥ 2.10
4. Compute link velocity from discharge / depth
5. Advance RiverBedDynamics by one timestep
6. Apply zero-gradient ghost-cell BC at the upstream boundary

Bed profiles are saved every **1 hour** to capture the rapid initial infilling.

In [ ]:
print(f"Running baseline simulation ({sim_max_t / 86400:.0f} days)...")

t = 0
progress0 = 0
save_counter = copy.deepcopy(save_interval)

# Snapshot storage
z_snapshots = [z[sample_node_id].copy()]
t_snapshots = [0.0]

while t < sim_max_t:
    # Previous-step velocity (from already-remapped link depths)
    rbd._surface_water__velocity_prev_time_link = (
        rmg["link"]["surface_water__discharge"]
        / rmg["link"]["surface_water__depth"]
    )

    of.overland_flow(dt=max_dt)

    # Critical: remap node depths → links after every OverlandFlow step
    rmg["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg, "surface_water__depth"
    )
    rmg["link"]["surface_water__velocity"] = (
        rmg["link"]["surface_water__discharge"]
        / rmg["link"]["surface_water__depth"]
    )

    rbd._grid._dt = max_dt
    rbd.run_one_step()

    # Zero-gradient upstream ghost cells
    for i in calc_node_id:
        rmg["node"]["topographic__elevation"][i] = rmg["node"][
            "topographic__elevation"
        ][i - n_col]

    # Save periodic snapshots
    save_counter -= max_dt
    if save_counter <= 0:
        z_snapshots.append(z[sample_node_id].copy())
        t_snapshots.append(t)
        save_counter = copy.deepcopy(save_interval)

    t += max_dt
    progress = int((t / sim_max_t) * 100)
    if progress > progress0 + 1:
        print(f"\r  Baseline: [{progress}%]", end="")
        progress0 = progress

# Final snapshot
z_snapshots.append(z[sample_node_id].copy())
t_snapshots.append(t)
z_baseline_final = z[sample_node_id].copy()

print("\n  Baseline complete.\n")


## 6. Baseline Results

### Bed profile evolution and infilling curve

The left panel shows the bed profile at selected times.  The hole fills rapidly in the first ~6 hours, then progressively more slowly as the bed approaches its original shape.  This is characteristic of a **diffusive relaxation process**: the rate of change is proportional to the curvature of the perturbation, which decreases as the hole fills.

The right panel quantifies this by tracking the remaining hole depth at x = 750 m over time.  The exponential-like decay confirms the diffusive character of the infilling process.

In [ ]:
# ── Figure 3 — Time evolution of bed profile ─────────────────────────
n_snaps = len(z_snapshots)

# Subsample: show every 6th snapshot (≈ every 6 hours) for clarity
step = max(1, n_snaps // 12)
show_idx = list(range(0, n_snaps, step))
if (n_snaps - 1) not in show_idx:
    show_idx.append(n_snaps - 1)
colors_sub = cm.viridis(np.linspace(0.0, 1.0, len(show_idx)))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(x_profile, z_profile_original, "k--", lw=1, alpha=0.4,
        label="Original (no hole)")
for ci, idx in enumerate(show_idx):
    zs = z_snapshots[idx]
    ts = t_snapshots[idx]
    ax.plot(x_profile, zs, color=colors_sub[ci], lw=1.2,
            label=f"t = {ts / 3600:.0f} h")
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Bed elevation [m]")
ax.set_title("Bed profile evolution (baseline)")
ax.legend(fontsize=7, ncol=2, loc="upper left")
ax.set_xlim(400, 1100)

# Right panel: remaining hole depth at the centre vs time
ax = axes[1]
i_ctr = np.argmin(np.abs(x_profile - hole_center_x))
depth_series = [z_profile_original[i_ctr] - zs[i_ctr] for zs in z_snapshots]
t_hours = [ts / 3600 for ts in t_snapshots]
ax.plot(t_hours, depth_series, "b-o", ms=3, lw=1.5)
ax.axhline(0, color="k", lw=0.8, ls=":")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Remaining hole depth [m]")
ax.set_title(f"Hole infilling at x = {hole_center_x:.0f} m")

plt.tight_layout()
plt.savefig("fig_t3_03_baseline_evolution.png", dpi=150)
plt.show()
print("Saved: fig_t3_03_baseline_evolution.png")

### Net erosion and deposition pattern

This figure shows the cumulative bed elevation change over the 3-day run.  The **deposition** (brown) occurs primarily on the upstream side of the hole, where the transport capacity drops as water deepens over the depression.  **Erosion** (blue) is visible downstream of the hole, where the flow accelerates off the downstream lip and entrains material from the undisturbed bed.

In [ ]:
# ── Figure 4 — Net erosion / deposition ──────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
dz_net = z_baseline_final - z_profile_with_hole
ax.fill_between(
    x_profile, 0, dz_net, where=(dz_net >= 0),
    color="sienna", alpha=0.6, label="Deposition",
)
ax.fill_between(
    x_profile, 0, dz_net, where=(dz_net < 0),
    color="steelblue", alpha=0.6, label="Erosion",
)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Elevation change [m]")
ax.set_title(f"Net bed change after {sim_max_t / 86400:.0f} days (baseline)")
ax.legend()
ax.set_xlim(0, 1500)
plt.tight_layout()
plt.savefig("fig_t3_04_net_change_baseline.png", dpi=150)
plt.show()
print("Saved: fig_t3_04_net_change_baseline.png\n")

## 7. Diffusion Run — Setup and Execution

To ensure a clean comparison, we re-initialise from scratch: fresh grid, same hole, same water surface initialisation.  The only difference is the activation of **gravitational bed diffusion** with μ = 0.5 (nonlinear mode).

The diffusion term adds a second contribution to the Exner equation:

$$(1 - \lambda_p) \frac{\partial z}{\partial t} = -\nabla \cdot \mathbf{q}_b + \nabla \cdot (D \nabla z)$$

where the diffusion coefficient is $D = |q_b| / \mu$.  At the steep edges of the hole, $|\nabla z|$ is large, so the diffusive flux actively redistributes sediment from the lips into the depression.  We expect the diffusion run to show:

- **Faster smoothing** of the sharp slope breaks at the hole edges
- **Slightly faster infilling** at the hole centre
- A more **symmetric** residual depression (advection alone creates asymmetry)

In [ ]:
print("--- Re-initialising for diffusion run ---")

t_d = 0

with open(zDEM) as f:
    rmg_d = load_esri_ascii(f, name="topographic__elevation")
z_d = rmg_d.at_node["topographic__elevation"]

rmg_d.add_zeros("surface_water__depth", at="node")
rmg_d.set_watershed_boundary_condition_outlet_id([1, 2], z_d, 45.0)

# Save original, then dig the same hole
z_d_original = z_d.copy()
hole_profile_d = hole_depth * np.exp(
    -((rmg_d.y_of_node - hole_center_y) ** 2) / (2.0 * hole_sigma ** 2)
)
z_d -= hole_profile_d
closed_d = np.where(rmg_d.status_at_node == 4)[0]
z_d[closed_d] = z_d_original[closed_d]

# Initialise smooth water surface over hole
h_d = np.maximum(wse_equilibrium - z_d, 0.001)
rmg_d.at_node["surface_water__depth"][:] = h_d
rmg_d["node"]["topographic__elevation_original"] = z_d.copy()

of_d = OverlandFlow(
    rmg_d, h_init=0.001, mannings_n=n_manning, rainfall_intensity=0.0,
)
of_d._rainfall_intensity = np.zeros_like(z_d, dtype=float)
of_d._rainfall_intensity[in_n] = 0.02

# Restore our water depth after OverlandFlow.__init__ reset it
rmg_d.at_node["surface_water__depth"][:] = h_d

rmg_d.add_zeros("surface_water__velocity", at="node")
rmg_d.add_zeros("surface_water__velocity", at="link")
rmg_d["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
    rmg_d, "surface_water__depth"
)

fixed_d = np.zeros(rmg_d.number_of_nodes)
fixed_d[fixed_outlet_nodes] = 1
fixed_d[fixed_upstream_nodes] = 1

calc_node_id_d = np.reshape(ghost_cell_ids, (n_row_ghost, n_col))

rbd_d = RiverBedDynamics(
    rmg_d,
    gsd=gsd,
    outlet_boundary_condition="fixedValue",
    bed_surf__elev_fix_node=fixed_d,
    check_advective_cfl=False,
    use_bed_diffusion=True,          # ← Gravitational diffusion ON
    bed_diffusion_mu=0.5,            # ← Talmon friction parameter
)

print(f"  bed_diffusion_mode : {rbd_d._bed_diffusion_mode}")
print(f"  bed_diffusion_mu   : {rbd_d._bed_diffusion_mu}")

### Hydraulic adjustment for the diffusion run (30 minutes)

Same brief adjustment period as the baseline — the flow must adapt to the hole geometry.

In [ ]:
# Brief hydraulic adjustment (30 min)
print("  Hydraulic adjustment (30 min)...")
t_adj = 0
while t_adj < 1800:
    of_d.overland_flow(dt=max_dt)
    rmg_d["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_d, "surface_water__depth"
    )
    t_adj += max_dt
rmg_d["link"]["surface_water__velocity"] = (
    rmg_d["link"]["surface_water__discharge"]
    / rmg_d["link"]["surface_water__depth"]
)
print("  Adjustment complete.\n")

### Run the diffusion simulation (3 days)

The simulation loop is identical to the baseline except that `rbd_d.run_one_step()` now internally calls `bed_diffusion()` before `update_bed_elevation()`.  Profiles are saved every 1 hour, matching the baseline.

In [ ]:
# ── Main diffusion loop ──────────────────────────────────────────────
print(f"  Running diffusion simulation ({sim_max_t / 86400:.0f} days)...")

z_snapshots_d = [z_d[sample_node_id].copy()]
t_snapshots_d = [0.0]
save_counter_d = copy.deepcopy(save_interval)
progress0 = 0

while t_d < sim_max_t:
    rbd_d._surface_water__velocity_prev_time_link = (
        rmg_d["link"]["surface_water__discharge"]
        / rmg_d["link"]["surface_water__depth"]
    )
    of_d.overland_flow(dt=max_dt)
    rmg_d["link"]["surface_water__depth"] = map_mean_of_link_nodes_to_link(
        rmg_d, "surface_water__depth"
    )
    rmg_d["link"]["surface_water__velocity"] = (
        rmg_d["link"]["surface_water__discharge"]
        / rmg_d["link"]["surface_water__depth"]
    )

    rbd_d._grid._dt = max_dt
    rbd_d.run_one_step()

    for i in calc_node_id_d:
        rmg_d["node"]["topographic__elevation"][i] = rmg_d["node"][
            "topographic__elevation"
        ][i - n_col]

    save_counter_d -= max_dt
    if save_counter_d <= 0:
        z_snapshots_d.append(z_d[sample_node_id].copy())
        t_snapshots_d.append(t_d)
        save_counter_d = copy.deepcopy(save_interval)

    t_d += max_dt
    progress = int((t_d / sim_max_t) * 100)
    if progress > progress0 + 1:
        print(f"\r  Diffusion: [{progress}%]", end="")
        progress0 = progress

z_snapshots_d.append(z_d[sample_node_id].copy())
t_snapshots_d.append(t_d)
z_diff_final = z_d[sample_node_id].copy()

print("\n  Diffusion simulation complete.\n")

## 8. Comparison — Baseline vs Diffusion

### Final profiles and diffusion effect

The left panel overlays the final profiles from both runs.  The right panel isolates the diffusion effect (diffusion − baseline).  Look for:

- **Positive values** at the hole centre: diffusion fills the hole slightly more than advection alone.
- **Negative values** at the edges: diffusion removes material from the lips where the slope is steepest.

This redistribution pattern is the signature of the Talmon et al. (1995) gravitational deflection of bedload trajectories.

In [ ]:
# ── Figure 5 — Final profiles: baseline vs diffusion ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(x_profile, z_profile_original, "k--", lw=1, alpha=0.4,
        label="Original (no hole)")
ax.plot(x_profile, z_profile_with_hole, "k:", lw=1, alpha=0.4,
        label="Initial (with hole)")
ax.plot(x_profile, z_baseline_final, "b-", lw=2,
        label="Baseline (no diffusion)")
ax.plot(x_profile, z_diff_final, "g--", lw=2,
        label="With diffusion (μ = 0.5)")
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Bed elevation [m]")
ax.set_title(f"Final profiles after {sim_max_t / 86400:.0f} days")
ax.legend(fontsize=9)
ax.set_xlim(400, 1100)

ax = axes[1]
diff_effect = z_diff_final - z_baseline_final
ax.plot(x_profile, diff_effect, "g-", lw=2)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Elevation difference [m]")
ax.set_title("Diffusion effect (diffusion − baseline)")
ax.set_xlim(400, 1100)

plt.tight_layout()
plt.savefig("fig_t3_05_baseline_vs_diffusion.png", dpi=150)
plt.show()
print("Saved: fig_t3_05_baseline_vs_diffusion.png")

# ── Figure 6 — Hole-filling rate comparison ──────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

i_ctr = np.argmin(np.abs(x_profile - hole_center_x))
depth_base = [z_profile_original[i_ctr] - zs[i_ctr] for zs in z_snapshots]
depth_diff = [z_profile_original[i_ctr] - zs[i_ctr] for zs in z_snapshots_d]
t_h_base = [ts / 3600 for ts in t_snapshots]
t_h_diff = [ts / 3600 for ts in t_snapshots_d]

ax.plot(t_h_base, depth_base, "b-o", ms=3, lw=1.5, label="Baseline")
ax.plot(t_h_diff, depth_diff, "g-s", ms=3, lw=1.5, label="Diffusion (μ = 0.5)")
ax.axhline(0, color="k", lw=0.8, ls=":")
ax.set_xlabel("Time [hours]")
ax.set_ylabel("Remaining hole depth [m]")
ax.set_title(f"Hole-filling rate at x = {hole_center_x:.0f} m")
ax.legend()
plt.tight_layout()
plt.savefig("fig_t3_06_hole_filling_rate.png", dpi=150)
plt.show()
print("Saved: fig_t3_06_hole_filling_rate.png\n")

## 9. Early Evolution — Hourly Comparison (t = 0–6 hours)

The infilling curve above shows that the most dramatic changes occur in the first few hours.  This section zooms into the **first 6 hours** at **hourly resolution** to reveal how the two methods differ during this critical early phase.

The left panel overlays the baseline profiles (solid blue shades) with the diffusion profiles (dashed green shades) at t = 0, 1, 2, 3, 4, 5, and 6 hours.  The right panel shows the difference (diffusion − baseline) at each hour, highlighting where and when diffusion has its greatest effect.

In [ ]:
# ── Figure 7 — Hourly comparison, t = 0–6 h ──────────────────────────
# Extract hourly snapshots (t_snapshots are in seconds, saved every 3600 s)
hours_to_show = [0, 1, 2, 3, 4, 5, 6]

# Build index lookup: find the snapshot closest to each target hour
def get_snapshot_at_hour(t_list, z_list, target_h):
    target_s = target_h * 3600.0
    idx = np.argmin(np.abs(np.array(t_list) - target_s))
    return z_list[idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Left panel: overlaid profiles ─────────────────────────────────
ax = axes[0]
ax.plot(x_profile, z_profile_original, "k--", lw=1, alpha=0.3,
        label="Original bed")

blues = cm.Blues(np.linspace(0.3, 1.0, len(hours_to_show)))
greens = cm.Greens(np.linspace(0.3, 1.0, len(hours_to_show)))

for ci, h in enumerate(hours_to_show):
    zb = get_snapshot_at_hour(t_snapshots, z_snapshots, h)
    zd = get_snapshot_at_hour(t_snapshots_d, z_snapshots_d, h)
    lbl_b = f"Baseline t={h}h" if h in [0, 3, 6] else None
    lbl_d = f"Diffusion t={h}h" if h in [0, 3, 6] else None
    ax.plot(x_profile, zb, "-", color=blues[ci], lw=1.5, label=lbl_b)
    ax.plot(x_profile, zd, "--", color=greens[ci], lw=1.5, label=lbl_d)

ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Bed elevation [m]")
ax.set_title("Early hole evolution: baseline (solid) vs diffusion (dashed)")
ax.legend(fontsize=8, loc="upper left")
ax.set_xlim(500, 1000)

# ── Right panel: difference at each hour ─────────────────────────
ax = axes[1]
diff_colors = cm.copper(np.linspace(0.2, 1.0, len(hours_to_show)))

for ci, h in enumerate(hours_to_show):
    if h == 0:
        continue  # difference is zero at t=0
    zb = get_snapshot_at_hour(t_snapshots, z_snapshots, h)
    zd = get_snapshot_at_hour(t_snapshots_d, z_snapshots_d, h)
    ax.plot(x_profile, zd - zb, "-", color=diff_colors[ci], lw=1.5,
            label=f"t = {h} h")

ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Streamwise distance [m]")
ax.set_ylabel("Elevation difference [m]")
ax.set_title("Diffusion − Baseline (hourly, t = 1–6 h)")
ax.legend(fontsize=8)
ax.set_xlim(500, 1000)

plt.tight_layout()
plt.savefig("fig_t3_07_early_hourly_comparison.png", dpi=150)
plt.show()
print("Saved: fig_t3_07_early_hourly_comparison.png\n")

## 10. Post-Run Diagnostics

The table below reports the GSD numerical health (residual before renormalisation) and the CFL stability limits at the end of each run.  Both runs should show residuals well below the default threshold (10⁻³) and CFL limits comfortably above the timestep used (dt = 1 s).

In [ ]:
print("=" * 60)
print("POST-RUN DIAGNOSTICS")
print("=" * 60)

for label, r in [("Baseline (Euler)", rbd), ("Diffusion (Euler)", rbd_d)]:
    print(f"\n--- {label} ---")
    print(f"  GSD residual max  : {r._bed_surf__gsd_residual_max:.2e}")
    print(f"  GSD residual mean : {r._bed_surf__gsd_residual_mean:.2e}")
    dt_adv = r.calc_max_stable_dt_advective(safety=0.5)
    dt_diff_cfl = r.calc_max_stable_dt_diffusive(safety=0.5)
    dt_safe = r.calc_max_stable_dt(safety=0.5)
    print(f"  CFL advective  (safety=0.5) : {dt_adv:.2f} s")
    print(f"  CFL diffusive  (safety=0.5) : {dt_diff_cfl:.2f} s")
    print(f"  Recommended dt (safety=0.5) : {dt_safe:.2f} s  "
          f"[dt used = {max_dt} s]")
    status = "OK" if max_dt <= dt_safe else "ABOVE LIMIT"
    print(f"  Status: {status}")

print("\n" + "=" * 60)
print("Tutorial 3 complete.")
print("=" * 60)